In [ ]:
import os

os.environ['TUSHARE_KEY'] = 'XXX'
from detonator import make_db_connection
from datetime import datetime
from dataminer.models import TickerDailyInfo

make_db_connection()

from dataminer import Indicators
import tushare as ts



In [ ]:
from detonator import mongo_2_df
import pandas as pd
TICKER = 'TSLA'
dailies = TickerDailyInfo.objects(ticker=TICKER).order_by('-trade_date')
dailies_df = mongo_2_df(dailies)
dailies_df['trade_date'] = pd.to_datetime(dailies_df['trade_date'], format='%Y,%m,%d,%H,%M,%S,%f')
dailies_df.set_index('trade_date', inplace=True)
dailies_df.sort_index(ascending=True, inplace=True)
print(dailies_df.shape)
dailies_df = dailies_df[['ticker','open', 'high', 'low', 'close', 'volume']]
dailies_df.dropna(inplace=True)
dailies_df.to_csv(f'{TICKER}.csv')
df = pd.read_csv(f'{TICKER}.csv', index_col=0, parse_dates=True)
df


In [ ]:
import os

from datetime import datetime

from yfinance import Ticker

aapl = Ticker('LI')
his = aapl.history(start=datetime(year=2024, month=1, day=6), end='2024-01-13')

In [ ]:
his

In [ ]:
his = aapl.history(period='5d', interval='1m', raise_errors=True)
his

In [ ]:
his.info()

In [ ]:
from dataminer.models import IndexTickers

queries = {
    'index_name': 'spx',
    'as_of_date__gte': '20240101'
}
it = IndexTickers.objects(__raw__ = queries)

In [ ]:
it.first().tickers

In [ ]:
import pandas as pd

csvs = pd.read_csv('https://www.ishares.com/us/products/239708/ishares-russell-1000-value-etf/1467271812596.ajax?fileType=csv&fileName=IWD_holdings&dataType=fund', skiprows = 9)
csvs = csvs[csvs['Asset Class']=='Equity']

In [ ]:
csvs

In [ ]:
import requests
from bs4 import BeautifulSoup
import os
from urllib.parse import urljoin


import pandas as pd
from pandas import DataFrame


def get_ishares_holdings_link(url):
    """
    Requests a given iShares ETF page and extracts the "Detailed Holdings and Analytics"
    link's label and href.

    Args:
        url (str): The URL of the iShares ETF page.

    Returns:
        tuple: A tuple containing (link_label, href_link) if found,
               otherwise (None, None).
    """
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }

    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)
    except requests.exceptions.RequestException as e:
        print(f"Error requesting the URL: {e}")
        return None, None

    soup = BeautifulSoup(response.content, 'html.parser')

    # Find the <a> tag with the specific class and text content
    # We can use a dictionary to specify attributes and their values
    # The `string` argument can be used to match the text content
    link_tag = soup.find(
        'a',
        # class_='icon-xls-export',
        string='Detailed Holdings and Analytics'
    )

    if link_tag:
        link_label = link_tag.get_text(strip=True)
        href_link = link_tag.get('href')

        # iShares often provides relative URLs for these links.
        # We need to construct a full URL if it's relative.
        if href_link and not href_link.startswith(('http://', 'https://')):
            # Assume it's a relative path to the base domain
            base_url = url.split('/us/products/')[0] + '/'
            href_link = urljoin(base_url, href_link)

        return link_label, href_link
    else:
        print("Link 'Detailed Holdings and Analytics' not found on the page.")
        return None, None

def download_csv_from_link(csv_url:str, filename:str="holdings.csv"):
    """
    Downloads a CSV file from a given URL.

    Args:
        csv_url (str): The URL of the CSV file.
        filename (str): The name to save the downloaded CSV file as.
    """
    if not csv_url:
        print("No CSV URL provided for download.")
        return

    headers = {
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36'
    }

    print(f"Attempting to download CSV from: {csv_url}")
    try:
        csv_response = requests.get(csv_url, headers=headers, stream=True)
        csv_response.raise_for_status() # Check for HTTP errors

        # Ensure the directory exists if specified in filename
        output_dir = os.path.dirname(filename)
        if output_dir and not os.path.exists(output_dir):
            os.makedirs(output_dir)

        with open(filename, 'wb') as f:
            for chunk in csv_response.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"CSV file '{filename}' downloaded successfully!")
    except requests.exceptions.RequestException as e:
        print(f"Error downloading the CSV: {e}")
    except IOError as e:
        print(f"Error writing the CSV file to disk: {e}")

# --- Main execution ---
target_url = "https://www.ishares.com/us/products/239706/ishares-russell-1000-growth-etf"
target_url = 'https://www.ishares.com/us/products/239710/ishares-russell-2000-etf'

print(f"Requesting page: {target_url}")
label, href = get_ishares_holdings_link(target_url)

if label and href:
    print(f"Found Link Label: '{label}'")
    print(f"Found Href: '{href}'")

    # You can now proceed to download the CSV if needed
    # For demonstration, I'll save it as 'IWB_holdings.csv'

    csvs = pd.read_csv(href, skiprows = 9)
    csvs = csvs[csvs['Asset Class']=='Equity']
    print(csvs)
else:
    print("Failed to find the detailed holdings link.")